# UMAP Seed Sensitivity: How Much Does the Gravity Field Change?

Tests whether the Koopman aesthetic-gravity field computed in `trajectory_mapping_clean.ipynb` is a stable
feature of the data, or largely an artifact of the particular UMAP `random_state` used to fit it.

**Approach**
1. Rebuild the normalized real+game XY contour feature matrix (same preprocessing as `umap_projection.ipynb`,
   after the parent-vase and long/wide fixes).
2. Re-fit UMAP on that matrix several times, varying only `random_state`; every other hyperparameter
   (`n_neighbors=200`, `min_dist=0.02`) stays fixed, matching the production pipeline.
3. For each seed's embedding, rebuild the game trajectory data and run the Koopman gravity pipeline
   (`koopman_gravity.py`) end-to-end to get a gravity field for that seed.
4. Compare fields across seeds two ways:
   - **Visually** — one gravity-field plot per seed, each in its own native coordinate frame.
   - **Quantitatively** — Procrustes-align each seed's game-vase point cloud onto a reference seed
     (removing the arbitrary rotation/reflection/scale UMAP is free to pick), interpolate gravity fields
     onto the reference grid, and correlate them. Separately, compare the dominant Koopman eigenvalue
     across seeds — a rotation-invariant scalar summarizing how strong/persistent the dominant mode is.

**Note on runtime:** fitting UMAP on ~135k points with `n_neighbors=200` is slow, and this notebook does it
once per seed. Start with 2-3 seeds in `SEEDS` below and confirm timing before expanding.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap.umap_ as umap
from numpy.linalg import svd
from scipy.interpolate import griddata

from koopman_gravity import (
    add_parent_vase_mapping, build_trajectory_df, build_trajectory_edges,
    build_transition_matrices, build_rbf_grid, build_koopman_operator,
    extract_dominant_eigenvector, choose_gravity_orientation,
    evaluate_gravity_on_grid, plot_gravity_field,
)

## 1. Rebuild the normalized XY feature matrix

Same preprocessing as `umap_projection.ipynb`: filter game vases to exclude parent-vase duplicates,
then center + max-abs normalize every profile (real and game alike) before flattening to a feature vector.

In [ ]:
STRAT_XY_PATH = "../real_vase_data/profiles_stratified_120526.parquet"
ALL_REAL_XY_PATH = "../real_vase_data/all_profiles_120526.parquet"
GAME_XY_PATH = "../file_processing/processed/xy_data.parquet"
GAME_META_PATH = "umap_projections/game_vase_umap_projection.parquet"

In [ ]:
all_real_xy = pd.read_parquet(ALL_REAL_XY_PATH)
strat_xy_df = pd.read_parquet(STRAT_XY_PATH)
strat_gpp_nos = strat_xy_df["gpp_no"].unique()

real_xy_df = all_real_xy[all_real_xy["gpp_no"].isin(strat_gpp_nos)].rename(columns={"gpp_no": "vase_id"}).copy()
real_xy_df["source"] = "real"

game_xy_df = pd.read_parquet(GAME_XY_PATH)
game_xy_df["vase_id"] = (
    game_xy_df["participant_id"].astype(str)
    + "::"
    + game_xy_df["generation_key"].astype(str)
    + "::"
    + game_xy_df["vase"].astype(str)
)
game_xy_df["source"] = "game"
game_xy_df = game_xy_df[game_xy_df["is_parent_vase"] != True]

print(f"real_xy_df: {real_xy_df['vase_id'].nunique():,} vases")
print(f"game_xy_df: {game_xy_df['vase_id'].nunique():,} vases")

In [ ]:
def normalize_profile(x, y):
    """Center on the profile's own mean and scale by max absolute coordinate."""
    x = x - np.nanmean(x)
    y = y - np.nanmean(y)
    scale = max(np.nanmax(np.abs(x)), np.nanmax(np.abs(y)), 1e-9)
    return x / scale, y / scale


def build_xy_feature_matrix(df, id_col="vase_id", order_col="point_order", normalize=True):
    rows, ids, sources = [], [], []
    for (vase_id, source), grp in df.groupby([id_col, "source"], sort=False):
        grp_sorted = grp.sort_values(order_col) if order_col in grp.columns else grp
        x = grp_sorted["x"].to_numpy(dtype=np.float32)
        y = grp_sorted["y"].to_numpy(dtype=np.float32)
        if normalize:
            x, y = normalize_profile(x, y)
        rows.append(np.concatenate([x, y]))
        ids.append(vase_id)
        sources.append(source)
    return np.stack(rows), pd.DataFrame({"vase_id": ids, "source": sources})


xy_matrix_real, meta_real = build_xy_feature_matrix(real_xy_df)
xy_matrix_game, meta_game = build_xy_feature_matrix(game_xy_df)

xy_array = np.vstack([xy_matrix_real, xy_matrix_game])
xy_meta_df = pd.concat([meta_real, meta_game], ignore_index=True)

print(f"XY feature matrix shape: {xy_array.shape}")
print(xy_meta_df["source"].value_counts())

## 2. Load game trajectory metadata (fixed across seeds)

Parent/child lineage, generation, and `is_selected_vase` don't depend on the UMAP embedding — only
`umap_x`/`umap_y` change per seed. Load this metadata once from the already-corrected
`game_vase_umap_projection.parquet` (dropping its baked-in coordinates), then attach each seed's
coordinates to it inside the loop below.

**Prerequisite:** this assumes `umap_projection.ipynb` has already been re-run with the parent-vase-filter
and long-to-wide pivot fixes, so `game_vase_umap_projection.parquet` reflects one row per unique vase.

In [ ]:
game_meta_df = pd.read_parquet(GAME_META_PATH)
game_meta_df = game_meta_df[game_meta_df["source"] == "game"].copy()
game_meta_df = game_meta_df.drop(columns=[c for c in game_meta_df.columns if c.startswith("PC")], errors="ignore")
game_meta_df = game_meta_df.drop(columns=["umap_x", "umap_y", "umap_x_pc", "umap_y_pc"], errors="ignore")

# add_parent_vase_mapping() expects a 'generation' column like load_umap_data() would add
game_meta_df["generation"] = game_meta_df["vase_id"].str.extract(r"(gen_\d+)").iloc[:, 0]

required_cols = {"participant_id", "play_counter", "vase_id", "is_selected_vase", "generation"}
missing = required_cols - set(game_meta_df.columns)
assert not missing, f"game_meta_df missing required columns: {missing}"

print(f"game_meta_df: {game_meta_df.shape}, unique vase_id: {game_meta_df['vase_id'].nunique():,}")
game_meta_df.head()

## 3. Fit UMAP across multiple seeds and run the gravity pipeline for each

`run_gravity_for_embedding()` mirrors `koopman_gravity.gravity_pipeline()`, but also returns the raw
eigenvalue spectrum (not just the oriented dominant eigenvector) so we can compare eigenvalues across seeds
directly, and returns each seed's own `umap_bounds`/grid rather than assuming a shared one.

In [ ]:
SEEDS = [42, 0, 7, 21, 67, 99, 34, 56]  # extend once you've confirmed this runs in a reasonable time on your machine

UMAP_KWARGS = dict(n_components=2, n_neighbors=200, min_dist=0.02, metric="euclidean")

In [ ]:
def compute_umap_bounds_from_df(df, pad=0.02):
    x_min, x_max = df["umap_x"].min(), df["umap_x"].max()
    y_min, y_max = df["umap_y"].min(), df["umap_y"].max()
    pad_x, pad_y = (x_max - x_min) * pad, (y_max - y_min) * pad
    return dict(
        x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max,
        xlim=(x_min - pad_x, x_max + pad_x), ylim=(y_min - pad_y, y_max + pad_y),
    )


def run_gravity_for_embedding(embedding, n_grid=30, chunk_size=10_000):
    seed_xy_df = xy_meta_df.copy()
    seed_xy_df["umap_x"] = embedding[:, 0]
    seed_xy_df["umap_y"] = embedding[:, 1]

    game_umap = seed_xy_df.loc[seed_xy_df["source"] == "game", ["vase_id", "umap_x", "umap_y"]]
    df = game_meta_df.merge(game_umap, on="vase_id", how="inner")

    df = add_parent_vase_mapping(df)
    trajectory_df = build_trajectory_df(df)
    trajectory_edges = build_trajectory_edges(trajectory_df)
    Z, Z_prime, pairs_df = build_transition_matrices(trajectory_edges, trajectory_df)

    umap_bounds = compute_umap_bounds_from_df(seed_xy_df)
    centres, grid_x, grid_y, sigma = build_rbf_grid(umap_bounds, n_grid)
    K = build_koopman_operator(Z, Z_prime, centres, sigma, chunk_size)
    evals, evecs, idx_dom, dom_evec = extract_dominant_eigenvector(K)
    oriented_evec = choose_gravity_orientation(trajectory_df, dom_evec, centres, sigma, chunk_size)
    phi_2D, U, V = evaluate_gravity_on_grid(grid_x, grid_y, oriented_evec, centres, sigma)

    return dict(
        embedding_game=game_umap.reset_index(drop=True),
        trajectory_df=trajectory_df,
        umap_bounds=umap_bounds,
        grid_x=grid_x, grid_y=grid_y, centres=centres, sigma=sigma,
        phi_2D=phi_2D, U=U, V=V,
        eigenvalues=evals, dominant_eigenvalue=evals[idx_dom],
    )

In [ ]:
results = {}
for seed in SEEDS:
    print(f"--- seed={seed} ---")
    reducer = umap.UMAP(random_state=seed, **UMAP_KWARGS)
    embedding = reducer.fit_transform(xy_array)
    results[seed] = run_gravity_for_embedding(embedding)
    print(f"  dominant eigenvalue: {results[seed]['dominant_eigenvalue']:.4f}")

## 4. Compare dominant eigenvalues across seeds

The dominant eigenvalue (closest to 1) reflects how strong/persistent the slowest-decaying mode of the
estimated dynamics is. It's a scalar derived from the Koopman operator, not tied to any particular
coordinate orientation — so if it comes out similar across seeds, that's independent evidence the gravity
field's *strength* is stable, even before we check its *shape* below.

In [ ]:
eig_summary = pd.DataFrame({
    "seed": SEEDS,
    "dominant_eigenvalue_real": [results[s]["dominant_eigenvalue"].real for s in SEEDS],
    "dominant_eigenvalue_imag": [results[s]["dominant_eigenvalue"].imag for s in SEEDS],
    "dominant_eigenvalue_abs": [abs(results[s]["dominant_eigenvalue"]) for s in SEEDS],
})
eig_summary

## 5. Visual comparison: gravity field per seed (native coordinates)

Each plot is in its own seed's coordinate frame, so a rotated/reflected but otherwise identical field
is expected and not a sign of instability — see the aligned comparison in section 6 for the real test.

In [ ]:
for seed in SEEDS:
    r = results[seed]
    plot_gravity_field(
        r["grid_x"], r["grid_y"], r["phi_2D"], r["U"], r["V"],
        r["umap_bounds"]["xlim"], r["umap_bounds"]["ylim"],
        title=f"Gravity Field (seed={seed})",
    )

## 6. Quantitative comparison: Procrustes-align embeddings, then correlate gravity fields

UMAP is free to rotate, reflect, and rescale its output between runs without changing the underlying
structure — so comparing raw coordinates or raw grids across seeds would confuse harmless reorientation
with genuine instability. Orthogonal Procrustes finds the best rotation/reflection/uniform-scale that maps
one seed's game-vase point cloud onto a reference seed's; applying that same transform to the gravity grid
and interpolating onto the reference grid lets us correlate the fields on equal footing. What's left over
after removing the rigid transform is genuine, non-rigid seed-to-seed variation.

In [ ]:
def orthogonal_procrustes_align(source_pts, target_pts):
    """Find R, scale, t minimizing ||scale * source_pts @ R + t - target_pts||
    (rotation/reflection + uniform scale + translation)."""
    src_mean, tgt_mean = source_pts.mean(axis=0), target_pts.mean(axis=0)
    src, tgt = source_pts - src_mean, target_pts - tgt_mean
    U_, S_, Vt_ = svd(src.T @ tgt)
    R = U_ @ Vt_
    scale = S_.sum() / (src ** 2).sum()
    t = tgt_mean - scale * src_mean @ R
    return R, scale, t


def apply_transform(points, R, scale, t):
    return scale * points @ R + t

In [ ]:
ref_seed = SEEDS[0]
ref = results[ref_seed]
ref_points_df = ref["embedding_game"].set_index("vase_id")

comparison_rows = []
for seed in SEEDS:
    r = results[seed]
    seed_points_df = r["embedding_game"].set_index("vase_id")
    common_ids = ref_points_df.index.intersection(seed_points_df.index)
    ref_pts = ref_points_df.loc[common_ids, ["umap_x", "umap_y"]].to_numpy()
    seed_pts = seed_points_df.loc[common_ids, ["umap_x", "umap_y"]].to_numpy()

    if seed == ref_seed:
        aligned_centres, aligned_phi = r["centres"], r["phi_2D"]
    else:
        R, scale, t = orthogonal_procrustes_align(seed_pts, ref_pts)
        aligned_centres = apply_transform(r["centres"], R, scale, t)
        aligned_phi = r["phi_2D"]

    # Interpolate this seed's (aligned) phi values onto the reference grid
    interp_phi = griddata(
        aligned_centres, aligned_phi.ravel(),
        (ref["grid_x"], ref["grid_y"]),
        method="linear",
    )
    valid = ~np.isnan(interp_phi) & ~np.isnan(ref["phi_2D"])
    corr = (
        np.corrcoef(interp_phi[valid].ravel(), ref["phi_2D"][valid].ravel())[0, 1]
        if valid.sum() > 10 else np.nan
    )

    comparison_rows.append(dict(
        seed=seed,
        n_matched_grid_points=int(valid.sum()),
        phi_correlation_vs_ref=corr,
    ))

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## 7. Interpretation

- **`dominant_eigenvalue_abs`** close to 1 and consistent across seeds → a persistent, slow-decaying
  gravity mode reliably exists regardless of seed. Large swings suggest the "mode" itself is noisy.
- **`phi_correlation_vs_ref`** near 1 → after removing arbitrary rotation/reflection/scale, the gravity
  field's *shape* is essentially the same across seeds (the field is a real property of the data, not a
  fitting artifact). Values well below 1 (especially toward 0) mean seed-to-seed changes are genuinely
  non-rigid — different embeddings are warping the manifold differently enough to change where gravity
  wells/sources sit, and any single seed's field shouldn't be treated as ground truth without averaging
  or otherwise accounting for that variance.

**Next steps if variance turns out to be high:** increase `SEEDS` for a more robust estimate, consider
averaging `phi_2D` across aligned seeds before drawing conclusions from the gravity field, or reduce
`n_neighbors` sensitivity by checking whether the instability is specific to this dataset's cluster
geometry (e.g. via the same alignment analysis restricted to a single dense sub-region).